# Adım 3b: Spark Structured Streaming — Kafka → Delta Lake Bronze
**Kişi 2 sorumluluğu** — `feature/spark-eda` branch

> Docker servisleri çalışıyor olmalı: `docker-compose up -d`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp, to_date
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

KAFKA_BOOTSTRAP       = 'localhost:9092'
KAFKA_TOPIC           = 'climate-data'
BRONZE_STREAMING_PATH = './delta_lake/bronze_streaming'
CHECKPOINT_PATH       = './checkpoints/bronze_streaming'
STREAM_DURATION_SEC   = 120

MESSAGE_SCHEMA = StructType([
    StructField('timestamp',              StringType()),
    StructField('station_id',             StringType()),
    StructField('city_name',              StringType()),
    StructField('date',                   StringType()),
    StructField('season',                 StringType()),
    StructField('avg_temp_c',             DoubleType()),
    StructField('min_temp_c',             DoubleType()),
    StructField('max_temp_c',             DoubleType()),
    StructField('precipitation_mm',       DoubleType()),
    StructField('snow_depth_mm',          DoubleType()),
    StructField('avg_wind_speed_kmh',     DoubleType()),
    StructField('avg_sea_level_pres_hpa', DoubleType()),
    StructField('sunshine_total_min',     DoubleType()),
])

def create_spark():
    return (
        SparkSession.builder
        .appName('ClimateStructuredStreaming')
        .master('local[*]')
        .config('spark.sql.extensions',
                'io.delta.sql.DeltaSparkSessionExtension')
        .config('spark.sql.catalog.spark_catalog',
                'org.apache.spark.sql.delta.catalog.DeltaCatalog')
        .config('spark.jars.packages',
                'io.delta:delta-core_2.12:2.4.0,'
                'org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1')
        .getOrCreate()
    )

spark = create_spark()
spark.sparkContext.setLogLevel('WARN')
print('[STREAMING] Spark session hazir.')

## Kafka → Delta Lake Streaming Sorgusu

In [ ]:
def start_streaming(spark):
    print(f'[STREAMING] Kafka topic okunuyor: {KAFKA_TOPIC}')
    raw_stream = (
        spark.readStream
        .format('kafka')
        .option('kafka.bootstrap.servers', KAFKA_BOOTSTRAP)
        .option('subscribe', KAFKA_TOPIC)
        .option('startingOffsets', 'latest')
        .option('failOnDataLoss', 'false')
        .load()
    )
    parsed = (
        raw_stream
        .select(from_json(col('value').cast('string'), MESSAGE_SCHEMA).alias('d'))
        .select(
            col('d.station_id'), col('d.city_name'),
            to_date(col('d.date')).alias('date'), col('d.season'),
            col('d.avg_temp_c'), col('d.min_temp_c'), col('d.max_temp_c'),
            col('d.precipitation_mm'), col('d.snow_depth_mm'),
            col('d.avg_wind_speed_kmh'), col('d.avg_sea_level_pres_hpa'),
            col('d.sunshine_total_min'),
            current_timestamp().alias('ingested_at'),
        )
    )
    query = (
        parsed.writeStream
        .format('delta')
        .outputMode('append')
        .option('checkpointLocation', CHECKPOINT_PATH)
        .option('path', BRONZE_STREAMING_PATH)
        .trigger(processingTime='10 seconds')
        .start()
    )
    print(f'[STREAMING] Hedef: {BRONZE_STREAMING_PATH}')
    print(f'[STREAMING] {STREAM_DURATION_SEC} saniye calisacak...')
    query.awaitTermination(STREAM_DURATION_SEC)
    query.stop()
    print('[STREAMING] Streaming tamamlandi.')
    try:
        count = spark.read.format('delta').load(BRONZE_STREAMING_PATH).count()
        print(f'[STREAMING] Bronze streaming: {count:,} kayit yazildi.')
    except Exception:
        print('[STREAMING] Bronze streaming tablosu bos veya henuz olusturulmadi.')

start_streaming(spark)
spark.stop()
print('Streaming pipeline tamamlandi.')